In [ ]:
from sympy import *

In [1]:
def hrs_min_sec(sec_val):
    hours=str(sec_val//(60**2))
    minutes=str((sec_val//60)%60)
    seconds=str(round(sec_val%60,0))
    if sec_val//(60**2)!=0:
        return(hours+' hrs '+minutes+' min '+seconds+' sec')
    if (sec_val//60)%60!=0:
        return(minutes+' min '+seconds+' sec')
    return(seconds+' sec')

In [2]:
def find_cochain_basis(ss):
    '''args: ss (spanning set), a list of cochains of the same homogeneous degree
       Returns: A list of cochains which are a basis for the subspace spanned by ss'''
    if len(ss)==0: return []
    if ss[0].deg=='UNKNOWN':
        ss[0].deg=cochain_deg(ss[0].deg)
    basis_set=set()
    for c in ss:
        for base_elt in c.coeff_dict:
            basis_set.add(base_elt)
    basis_list=list(basis_set)
    
    this_mat=zeros(len(ss),len(basis_list))
    for i in range(len(ss)):
        set_row(this_mat,i,coordinatize_cochain_in_basis(ss[i],basis_list))
    this_mat=this_mat.rref()[0]
    result=[list(this_mat.row(i)) for i in range(shape(this_mat)[0]) 
            if list(this_mat.row(i))!=[0]*len(this_mat.row(i))]   
    basis_cochains=[cochain({A:1}) for A in basis_list]
    return([coords_to_lin_comb(A,basis_cochains) for A in result])


In [3]:
def set_row(mat,rowNum,row):
    if type(row)==type(zeros(3,3)):
        rowList=list(row)
    else: 
        if type(row)==type([0]):
            rowList=row
        else: print('setRow error: arg row must be either matrix or list')
    if len(rowList)!=len(mat.row(0)):
        print('setRow error: mat.row() and row have differing lengths')
        return None
    for i in range(len(rowList)):
        mat[rowNum,i]=rowList[i]
        
def set_col(mat,colNum,col):
    if type(col)==type(zeros(2,2)):
        colList=list(col)
    else:
        if type(col)==type([0]):
            colList=col
        else: print('SetCol error: arg col must be either matrix or list')
            
    if len(colList)!=len(mat.col(0)):
        print('SetCol error: mat.col() and col have differing lengths')
        return None
    for i in range(len(colList)):
        mat[i,colNum]=colList[i]

In [4]:
def coords_to_lin_comb(basis,coords):
    '''args: basis, a list of symbols, and coords, a vector of the same length
       Returns: A LinComb corresponding to the vector coords
       NOTE: basis must be a list of symbols'''
    
    # Check if the lengths are the same:
    if len(basis)!=len(coords):
        print('coords_to_lin_comb error: |basis| and |coords| have different lengths')
        return None
    
    result=0
    for i in range(len(basis)):
        result=result+coords[i]*basis[i]
    return result

In [5]:
def coordinatize_cochain_in_basis(c,basis):
    '''c: a cochain object of homogeneous degree
       basis: a collection of tuples representing cochains
       returns the vector representation of c w.r.t basis as a list'''
    result=[0]*len(basis)
    for key in c.coeff_dict:
        if key in basis: result[basis.index(key)]=c.coeff_dict[key]
        else: print('coordinatize_cochain_in_basis error: cochain component',key,'not in basis')
    return result

In [6]:
def convert_T_symb_elt_to_cochain(se):
    '''se: a T_symb_elt object or a T_symb_basis object
       returns: a degree 0 cochain object corresponding to se'''
    if se==0: return cochain({})
    return cochain({(str(T_symb_basis[i]),):se.vec_rep[i] 
                    for i in range(len(T_symb_basis))})